# SmartBite YOLO26s-OBB ExpDate-2K Fine-Tuning After Brazil

This notebook fine-tunes the Brazil-trained SmartBite YOLO26s-OBB expiry-date detector on the local ExpDate-2K single-class OBB dataset.

Expected inputs:

1. Upload `data/expdate-2k-yolo-obb-split.zip` to Drive.
2. Finish the Brazilian fine-tuning run and copy its artifacts to Drive.
3. Use the Brazilian `weights/best.pt` as the starting checkpoint here.

The dataset zip is already single-class OBB (`0: expiry_date`) and has the 6 empty-label source images removed.


## Mount Drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Reset Working Directories


In [ ]:
!rm -rf /content/dataset /content/yolo_obb_dataset /content/output
!mkdir -p /content/dataset /content/yolo_obb_dataset /content/output


## Helpers


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run_live(cmd, env=None, cwd=None):
    cmd = [str(x) for x in cmd]
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


## Config

Upload the prepared dataset zip here:

```text
/content/drive/My Drive/sb-colab/expdate-2k-yolo-obb-split.zip
```

This notebook starts from the model produced by the Brazilian run:

```text
/content/drive/My Drive/sb-colab/yolo26s_obb_brazil_expdate_ft_date_due/weights/best.pt
```

If your Brazilian run name differs, change only `BASE_CHECKPOINT_DRIVE`.


In [ ]:
DATASET_ZIP = Path('/content/drive/My Drive/sb-colab/expdate-2k-yolo-obb-split.zip')
BASE_UNZIP_DIR = Path('/content/dataset')
YOLO_DATASET_ROOT = Path('/content/yolo_obb_dataset/expdate_2k_expiry_obb')

# Fine-tune from the YOLO26s-OBB checkpoint produced after Brazilian training.
BASE_CHECKPOINT_DRIVE = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_brazil_expdate_ft_date_due/weights/best.pt')
YOLO_MODEL_SOURCE = str(BASE_CHECKPOINT_DRIVE)

RUNS_PROJECT = Path('/content/output')
RUN_NAME = 'smartbite_yolo26s_obb_expdate2k_ft_after_brazil'
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_expdate2k_ft_after_brazil')
FINAL_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_expdate2k_ft_after_brazil.zip')

EXPECTED_COUNTS = {'train': 1339, 'valid': 363, 'test': 196}

EPOCHS = 80
IMGSZ = 1024
BATCH = 12
DEVICE = '0'  # set 'cpu' if no GPU
WORKERS = 2
PATIENCE = 20

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'
assert BASE_CHECKPOINT_DRIVE.exists(), f'Missing Brazil-trained checkpoint: {BASE_CHECKPOINT_DRIVE}'
print('DATASET_ZIP =', DATASET_ZIP)
print('YOLO_MODEL_SOURCE =', YOLO_MODEL_SOURCE)
print('RUN_NAME =', RUN_NAME)
print('FINAL_MODEL_DRIVE_DIR =', FINAL_MODEL_DRIVE_DIR)


## Unzip ExpDate-2K OBB Dataset


In [ ]:
run_live(['unzip', '-q', '-o', DATASET_ZIP, '-d', BASE_UNZIP_DIR])


def find_yolo_obb_dataset_root(base: Path) -> Path:
    candidates = []
    for yaml_path in base.rglob('data.yaml'):
        root = yaml_path.parent
        if (root / 'train' / 'images').exists() and (root / 'train' / 'labels').exists():
            candidates.append(root)
    assert candidates, 'Could not find YOLO OBB dataset root containing data.yaml and train/images.'
    return sorted(candidates, key=lambda p: len(str(p)))[0]


SOURCE_DATASET_ROOT = find_yolo_obb_dataset_root(BASE_UNZIP_DIR)
SOURCE_DATASET_YAML = SOURCE_DATASET_ROOT / 'data.yaml'
print('SOURCE_DATASET_ROOT =', SOURCE_DATASET_ROOT)
print(SOURCE_DATASET_YAML.read_text())

for split, expected in EXPECTED_COUNTS.items():
    image_count = len(list((SOURCE_DATASET_ROOT / split / 'images').glob('*')))
    label_paths = sorted((SOURCE_DATASET_ROOT / split / 'labels').glob('*.txt'))
    label_count = len(label_paths)
    empty_labels = [p for p in label_paths if not p.read_text(encoding='utf-8').strip()]
    print(split, 'images:', image_count, 'labels:', label_count, 'empty labels:', len(empty_labels))
    assert image_count == expected, f'{split} image count mismatch: {image_count} != {expected}'
    assert label_count == expected, f'{split} label count mismatch: {label_count} != {expected}'
    assert not empty_labels, f'{split} still has empty labels: {[p.name for p in empty_labels[:5]]}'

if YOLO_DATASET_ROOT.exists():
    shutil.rmtree(YOLO_DATASET_ROOT)
YOLO_DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(SOURCE_DATASET_ROOT, YOLO_DATASET_ROOT)
DATASET_YAML = YOLO_DATASET_ROOT / 'data.yaml'
print('Training dataset copied to:', YOLO_DATASET_ROOT)
print(DATASET_YAML.read_text())


## Install Ultralytics


In [ ]:
run_live([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'pyyaml'])

from ultralytics import YOLO

probe = YOLO(YOLO_MODEL_SOURCE)
print('Loaded checkpoint task:', getattr(probe, 'task', None))
print('Loaded checkpoint names:', getattr(probe, 'names', None))


## Fine-Tune YOLO26s-OBB From Brazilian Checkpoint


In [ ]:
train_script = f'''
import contextlib
import os
import sys
from ultralytics import YOLO

PRINT_EVERY = 50
state = {{"batch": 0}}


def log(msg):
    print(msg, file=sys.stderr, flush=True)


def on_train_batch_end(trainer):
    state["batch"] += 1
    if state["batch"] % PRINT_EVERY == 0:
        epoch = getattr(trainer, "epoch", 0) + 1
        epochs = getattr(trainer, "epochs", "?")
        loss_items = getattr(trainer, "loss_items", None)
        log(f"epoch {{epoch}}/{{epochs}} step {{state['batch']}} loss={{loss_items}}")


def on_fit_epoch_end(trainer):
    epoch = getattr(trainer, "epoch", 0) + 1
    metrics = getattr(trainer, "metrics", None)
    log(f"epoch {{epoch}} finished metrics={{metrics}}")


model = YOLO(r"{YOLO_MODEL_SOURCE}")
model.add_callback("on_train_batch_end", on_train_batch_end)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
    results = model.train(
        data=r"{DATASET_YAML}",
        epochs={EPOCHS},
        imgsz={IMGSZ},
        batch={BATCH},
        device=r"{DEVICE}",
        project=r"{RUNS_PROJECT}",
        name=r"{RUN_NAME}",
        workers={WORKERS},
        patience={PATIENCE},
        cache=False,
        plots=True,
        close_mosaic=10,
        verbose=False,
    )

log(results)
log("Training finished.")
'''

run_live([sys.executable, '-c', train_script])


## Validate Best Checkpoint


In [ ]:
best_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'best.pt'
last_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'last.pt'
assert best_pt.exists(), f'Missing best checkpoint: {best_pt}'
print('best_pt =', best_pt)
print('last_pt =', last_pt, last_pt.exists())

val_script = f'''
import json
from ultralytics import YOLO


def compact_metrics(metrics):
    return {{k: float(v) for k, v in metrics.results_dict.items()}}

model = YOLO(r"{best_pt}")
print('--- VAL ---')
metrics_val = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='val')
print(json.dumps(compact_metrics(metrics_val), indent=2))
print('--- TEST ---')
metrics_test = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='test')
print(json.dumps(compact_metrics(metrics_test), indent=2))
'''
run_live([sys.executable, '-c', val_script])


## Quick Prediction Preview


In [ ]:
preview_script = f'''
from pathlib import Path
from ultralytics import YOLO

model = YOLO(r"{best_pt}")
source = Path(r"{YOLO_DATASET_ROOT}") / 'test' / 'images'
results = model.predict(
    source=str(source),
    imgsz={IMGSZ},
    conf=0.05,
    device=r"{DEVICE}",
    project=r"{RUNS_PROJECT}",
    name=r"{RUN_NAME}_preview",
    save=True,
    max_det=20,
)
print('Preview images saved to:', Path(r"{RUNS_PROJECT}") / '{RUN_NAME}_preview')
print('Predicted images:', len(results))
'''
run_live([sys.executable, '-c', preview_script])


## Backup Artifacts To Drive


In [ ]:
src = RUNS_PROJECT / RUN_NAME
assert src.exists(), f'Missing run dir: {src}'
if FINAL_MODEL_DRIVE_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
FINAL_MODEL_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, FINAL_MODEL_DRIVE_DIR)
print('Saved YOLO26s-OBB artifacts to:', FINAL_MODEL_DRIVE_DIR)

if FINAL_ZIP_DRIVE.exists():
    FINAL_ZIP_DRIVE.unlink()
with zipfile.ZipFile(FINAL_ZIP_DRIVE, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(FINAL_MODEL_DRIVE_DIR.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(FINAL_MODEL_DRIVE_DIR.parent))
print('Saved zip:', FINAL_ZIP_DRIVE)
print('Best checkpoint:', FINAL_MODEL_DRIVE_DIR / 'weights' / 'best.pt')


## Notes

- This run is a second-stage fine-tune: original ExpDate-derived SmartBite model -> Brazilian dataset -> ExpDate-2K dataset.
- The dataset has no empty-label images; they were removed during local prep.
- Keep the final test64 benchmark frozen. The meaningful question is whether this checkpoint improves hard `IMG_*` cases, not whether it scores high on this dataset's own validation split.
- If Colab runs out of memory at `BATCH = 12`, lower it to `8` or `4` and rerun.
